<div align="center">

# MACHINE LEARNING IV
## Complete Assignment Project

**Student Name:** Atharv M Bennur  
**Roll Number:** 224010  
**Course:** Bachelor of Computer Applications

---

### Classification · Regression · Clustering · Prediction

</div>

> **Colab and local execution:** Each project has its own dataset-upload cell. In Google Colab, that cell uses `google.colab.files.upload()`. In local VS Code Jupyter, the same cell opens a CSV file picker. Upload/select only the dataset named in the current section. The notebook does not rely on fixed local paths or separate project files.

## Table of contents

1. [Part 1, Group A - Iris Classification](#part-1-group-a-iris-classification)
2. [Part 1, Group B - Car Price Regression](#part-1-group-b-car-price-regression)
3. [Part 1, Group C - Mall Customer Clustering](#part-1-group-c-mall-customer-clustering)
4. [Part 2 - Student Performance Prediction](#part-2-student-performance-prediction)
5. [Overall Conclusion](#overall-conclusion)

**Execution note:** Run cells sequentially within each section. This deliverable contains no pre-filled metrics or outputs; executing a section with its data generates the results.

<a id="part-1-group-a-iris-classification"></a>
# PART 1 - GROUP A: IRIS CLASSIFICATION
## Selected case C1 - Iris

### Objective and problem statement

Classify iris flowers from their four sepal and petal measurements. Compare Logistic Regression and K-Nearest Neighbors (KNN), tune KNN on training folds only, and evaluate both models on a held-out test set. The required accuracy target is at least 93%; report measured results honestly.

In [ ]:
import io
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid", palette="deep")

### Dataset upload

Run in Colab and select **Iris.csv**. This cell requests no other dataset.

In [ ]:
# Colab upload or local VS Code file picker; select Iris.csv.
from pathlib import Path

try:
    from google.colab import files
    uploaded = files.upload()
except ImportError:
    from tkinter import Tk, filedialog

    picker = Tk()
    picker.withdraw()
    selected_file = filedialog.askopenfilename(
        title="Select Iris.csv",
        filetypes=[("CSV files", "*.csv")],
    )
    picker.destroy()
    if not selected_file:
        raise FileNotFoundError("No CSV was selected.")
    uploaded = {Path(selected_file).name: Path(selected_file).read_bytes()}

expected_filename = 'Iris.csv'
if expected_filename not in uploaded:
    raise FileNotFoundError(
        f"Expected {expected_filename!r}. Rerun this cell and upload/select that exact CSV filename."
    )
df_iris_raw = pd.read_csv(
    io.BytesIO(uploaded[expected_filename])
)
print("Loaded:", expected_filename)

### Load and inspect

In [ ]:
print("Shape:", df_iris_raw.shape)
print("Columns:", df_iris_raw.columns.tolist())
display(df_iris_raw.dtypes.rename("dtype").to_frame())
display(df_iris_raw.head())
display(df_iris_raw.isna().sum().rename("missing").to_frame())
print("Exact duplicate rows:", int(df_iris_raw.duplicated().sum()))

### Data cleaning

Drop exact duplicates and rows missing required measurements/target. `Id` is an identifier, not a feature; it is excluded from modeling.

In [ ]:
iris_features = ["SepalLengthCm", "SepalWidthCm", "PetalLengthCm", "PetalWidthCm"]
iris_target = "Species"
iris_required = ["Id", *iris_features, iris_target]
iris_missing_columns = sorted(set(iris_required) - set(df_iris_raw.columns))
if iris_missing_columns:
    raise ValueError(f"Iris CSV is missing required columns: {iris_missing_columns}")
iris_duplicate_count = int(df_iris_raw.duplicated().sum())
df_iris = df_iris_raw.drop_duplicates().dropna(subset=iris_features + [iris_target]).copy()
print(f"Removed {iris_duplicate_count} exact duplicates and {len(df_iris_raw) - len(df_iris)} rows with missing required values.")
print("Id excluded from features:", "Id" not in iris_features)

### Exploratory data analysis

In [ ]:
display(df_iris["Species"].value_counts().rename_axis("Species").to_frame("count"))
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.countplot(data=df_iris, x="Species", hue="Species", legend=False, ax=axes[0])
axes[0].set(title="Iris Species Distribution", xlabel="Species", ylabel="Flower count")
sns.scatterplot(data=df_iris, x="PetalLengthCm", y="PetalWidthCm", hue="Species", s=60, ax=axes[1])
axes[1].set(title="Petal Measurements by Species", xlabel="Petal length (cm)", ylabel="Petal width (cm)")
plt.tight_layout()
plt.show()
fig, ax = plt.subplots(figsize=(7, 5))
sns.scatterplot(data=df_iris, x="SepalLengthCm", y="SepalWidthCm", hue="Species", s=60, ax=ax)
ax.set(title="Sepal Measurements by Species", xlabel="Sepal length (cm)", ylabel="Sepal width (cm)")
plt.tight_layout()
plt.show()

### Data preprocessing

Use only the four numeric flower measurements. Scaling is fitted inside each model pipeline on training data.

In [ ]:
X_iris = df_iris[iris_features].copy()
y_iris = df_iris[iris_target].copy()
print("Features:", X_iris.columns.tolist())
print("Target classes:", sorted(y_iris.unique()))

### Train-test split - 80:20, random_state=42

In [ ]:
X_iris_train, X_iris_test, y_iris_train, y_iris_test = train_test_split(
    X_iris, y_iris, test_size=0.20, random_state=42, stratify=y_iris
)
print("Training rows:", len(X_iris_train), "| Test rows:", len(X_iris_test))

### Model training

In [ ]:
iris_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
iris_logistic = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, random_state=42)),
])
iris_knn_search = GridSearchCV(
    Pipeline([("scaler", StandardScaler()), ("model", KNeighborsClassifier())]),
    {"model__n_neighbors": list(range(1, 21))},
    cv=iris_cv, scoring="accuracy", n_jobs=1, refit=True,
)
iris_logistic.fit(X_iris_train, y_iris_train)
iris_knn_search.fit(X_iris_train, y_iris_train)
iris_knn = iris_knn_search.best_estimator_
print("KNN selected k:", iris_knn_search.best_params_["model__n_neighbors"])
print("Best KNN training-fold CV accuracy:", round(iris_knn_search.best_score_, 4))

### Model evaluation

In [ ]:
iris_models = {"Logistic Regression": iris_logistic, "K-Nearest Neighbors": iris_knn}
iris_predictions = {name: model.predict(X_iris_test) for name, model in iris_models.items()}
iris_metrics = {}
for name, predicted in iris_predictions.items():
    iris_metrics[name] = {
        "Accuracy": accuracy_score(y_iris_test, predicted),
        "Precision (weighted)": precision_score(y_iris_test, predicted, average="weighted", zero_division=0),
        "Recall (weighted)": recall_score(y_iris_test, predicted, average="weighted", zero_division=0),
        "F1-score (weighted)": f1_score(y_iris_test, predicted, average="weighted", zero_division=0),
    }
iris_metrics_table = pd.DataFrame(iris_metrics).T
display(iris_metrics_table.style.format("{:.3f}").set_caption("Held-out test metrics"))
print("93% accuracy target:", "MET" if iris_metrics_table["Accuracy"].max() >= 0.93 else "NOT MET")

### Required plots - confusion matrix heatmaps

In [ ]:
iris_labels = sorted(y_iris.unique())
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, (name, predicted) in zip(axes, iris_predictions.items()):
    matrix = confusion_matrix(y_iris_test, predicted, labels=iris_labels)
    sns.heatmap(matrix, annot=True, fmt="d", cmap="Blues", xticklabels=iris_labels, yticklabels=iris_labels, ax=ax)
    ax.set(title=f"{name} - Confusion Matrix", xlabel="Predicted species", ylabel="Actual species")
plt.tight_layout()
plt.show()

### Three-line model comparison and interpretation

In [ ]:
iris_cv_logistic = cross_val_score(iris_logistic, X_iris_train, y_iris_train, cv=iris_cv, scoring="accuracy").mean()
iris_cv_scores = {"Logistic Regression": iris_cv_logistic, "K-Nearest Neighbors": iris_knn_search.best_score_}
iris_best_name = max(iris_cv_scores, key=iris_cv_scores.get)
print(f"1. Logistic Regression test accuracy: {iris_metrics_table.loc['Logistic Regression', 'Accuracy']:.1%}; training-only CV accuracy: {iris_cv_logistic:.1%}.")
print(f"2. Tuned KNN (k={iris_knn_search.best_params_['model__n_neighbors']}) test accuracy: {iris_metrics_table.loc['K-Nearest Neighbors', 'Accuracy']:.1%}; training-only CV accuracy: {iris_knn_search.best_score_:.1%}.")
print(f"3. {iris_best_name} is selected based on training cross-validation; the held-out test set is reserved for final evaluation.")

### Save best model and predict new flower measurements

In [ ]:
iris_model_path = "iris_best_model.joblib"
joblib.dump(iris_models[iris_best_name], iris_model_path)
iris_loaded_model = joblib.load(iris_model_path)
new_iris_flowers = pd.DataFrame([
    [5.1, 3.5, 1.4, 0.2], [6.4, 3.2, 4.5, 1.5], [6.7, 3.0, 5.2, 2.3],
], columns=iris_features)
new_iris_flowers["Predicted Species"] = iris_loaded_model.predict(new_iris_flowers[iris_features])
print("Saved and reloaded:", iris_model_path)
display(new_iris_flowers)

### Section conclusion

Use the displayed held-out metrics to determine whether the 93% threshold was met. KNN was tuned using training folds only; the selected model is saved and reloaded before predicting new measurements.

<a id="part-1-group-b-car-price-regression"></a>
# PART 1 - GROUP B: CAR PRICE REGRESSION
## Selected case R3 - Used-Car Price Prediction

### Objective

Estimate `selling_price` from the supplied vehicle attributes. Remove exact duplicates before splitting to reduce train/test overlap, compare engineered Linear Regression and regularized/tree candidates using training-only cross-validation, and report the chosen model on the fixed held-out test partition. Preserve the earlier R² ≈ 0.519 result as a historical baseline; the optimized workflow is a distinct, non-like-for-like evaluation.


In [ ]:
import io
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

sns.set_theme(style="whitegrid", palette="deep")

### Dataset upload - Car Dekho CSV

Run in Colab and select **CAR DETAILS FROM CAR DEKHO.csv**.

In [ ]:
# Colab upload or local VS Code file picker; select CAR DETAILS FROM CAR DEKHO.csv.
from pathlib import Path

try:
    from google.colab import files
    uploaded = files.upload()
except ImportError:
    from tkinter import Tk, filedialog

    picker = Tk()
    picker.withdraw()
    selected_file = filedialog.askopenfilename(
        title="Select CAR DETAILS FROM CAR DEKHO.csv",
        filetypes=[("CSV files", "*.csv")],
    )
    picker.destroy()
    if not selected_file:
        raise FileNotFoundError("No CSV was selected.")
    uploaded = {Path(selected_file).name: Path(selected_file).read_bytes()}

expected_filename = 'CAR DETAILS FROM CAR DEKHO.csv'
if expected_filename not in uploaded:
    raise FileNotFoundError(
        f"Expected {expected_filename!r}. Rerun this cell and upload/select that exact CSV filename."
    )
df_car_raw = pd.read_csv(
    io.BytesIO(uploaded[expected_filename])
)
print("Loaded:", expected_filename)

### Load and inspect

In [ ]:
print("Shape:", df_car_raw.shape)
print("Columns:", df_car_raw.columns.tolist())
display(df_car_raw.dtypes.rename("dtype").to_frame())
display(df_car_raw.isna().sum().rename("missing").to_frame())
print("Exact duplicate rows:", int(df_car_raw.duplicated().sum()))
display(df_car_raw.describe(include="all").T)
display(df_car_raw.head())

### Data cleaning and decisions

The raw file has 4,340 rows and 763 exact duplicate rows. Exact duplicates are removed **before** the train/test split so identical records cannot leak across partitions. Missing/non-positive targets are excluded; feature values are imputed inside training-fitted pipelines. IQR outliers are reported but retained because extreme prices and mileages may be valid. This CSV contains no mileage-economy, engine, torque, or max-power columns, so none are fabricated. We derive brand, a two-token model family, car age relative to the dataset's 2021 reference, and log kilometres driven.


In [ ]:
car_required = ["name", "year", "selling_price", "km_driven", "fuel", "seller_type", "transmission", "owner"]
car_missing_columns = sorted(set(car_required) - set(df_car_raw.columns))
if car_missing_columns:
    raise ValueError(f"Car CSV is missing required columns: {car_missing_columns}")

df_car = df_car_raw.copy()
for column in ["year", "selling_price", "km_driven"]:
    df_car[column] = pd.to_numeric(df_car[column], errors="coerce")
car_raw_rows = len(df_car)
car_duplicate_count = int(df_car.duplicated().sum())
df_car = df_car.drop_duplicates().copy()
car_rows_after_deduplication = len(df_car)
df_car = df_car.dropna(subset=["selling_price"])
df_car = df_car.loc[df_car["selling_price"] > 0].copy()
car_removed_target_rows = car_rows_after_deduplication - len(df_car)

df_car["name"] = df_car["name"].fillna("Unknown").astype(str).str.strip()
df_car["brand"] = df_car["name"].str.split().str[0].replace("", "Unknown")
df_car["model_family"] = df_car["name"].str.split().str[:2].str.join(" ").replace("", "Unknown")
df_car["car_age_2021"] = 2021 - df_car["year"]
df_car["log_km_driven"] = np.log1p(df_car["km_driven"].where(df_car["km_driven"] >= 0))

car_outlier_counts = {}
for column in ["selling_price", "km_driven", "year"]:
    values = df_car[column].dropna()
    q1, q3 = values.quantile([0.25, 0.75])
    iqr = q3 - q1
    car_outlier_counts[column] = int(((values < q1 - 1.5 * iqr) | (values > q3 + 1.5 * iqr)).sum())

display(df_car["brand"].value_counts().head(10).rename_axis("brand").to_frame("unique listings"))
print(f"Raw rows: {car_raw_rows:,} | Exact duplicates removed: {car_duplicate_count:,} | Usable unique rows: {len(df_car):,}")
print(f"Rows removed for missing/non-positive target: {car_removed_target_rows}")
print("IQR outliers retained:", car_outlier_counts)
print("Missing values after target cleaning:")
display(df_car.isna().sum().rename("missing").to_frame())


### Exploratory data analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
sns.histplot(data=df_car, x="selling_price", bins=40, color="steelblue", ax=axes[0, 0])
axes[0, 0].set(title="Selling-price distribution", xlabel="Selling price (INR)", ylabel="Unique listings")

car_year_median = df_car.groupby("year", dropna=True)["selling_price"].median()
axes[0, 1].plot(car_year_median.index, car_year_median.values, marker="o", color="darkorange")
axes[0, 1].set(title="Model year vs median selling price", xlabel="Model year", ylabel="Median selling price (INR)")

sns.scatterplot(data=df_car, x="km_driven", y="selling_price", hue="fuel", alpha=0.45, s=22, ax=axes[1, 0], legend=False)
axes[1, 0].set(title="Kilometres driven vs selling price", xlabel="Kilometres driven", ylabel="Selling price (INR)")

sns.boxplot(data=df_car, x="fuel", y="selling_price", color="lightsteelblue", ax=axes[1, 1])
axes[1, 1].set(title="Selling price by fuel type", xlabel="Fuel", ylabel="Selling price (INR)")
plt.tight_layout()
plt.show()
print("Prices are right-skewed; a log-target model is compared using training-only cross-validation.")


### Preprocessing and feature engineering

The feature set uses car age and `log1p(km_driven)` for numeric signal, plus fuel, seller, transmission, owner, brand, model family, and full model name. Numeric median imputation and categorical mode imputation/one-hot encoding are fit inside each training fold. We compare raw-target Linear Regression, log-target Linear Regression, tuned Ridge/Lasso, and tuned Extra Trees. Exact model-name features are retained only when training-only cross-validation supports them.


In [ ]:
car_numeric_features = ["car_age_2021", "log_km_driven"]
car_categorical_features = ["brand", "model_family", "name", "fuel", "seller_type", "transmission", "owner"]
car_feature_columns = car_numeric_features + car_categorical_features
X_car = df_car[car_feature_columns].copy()
y_car = df_car["selling_price"].copy()

def make_car_pipeline(log_target=False, estimator=None, min_frequency=3, categorical_features=None):
    selected_categories = categorical_features or car_categorical_features
    preprocessor = ColumnTransformer([
        ("numeric", SimpleImputer(strategy="median"), car_numeric_features),
        ("categorical", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=min_frequency)),
        ]), selected_categories),
    ])
    model = estimator if estimator is not None else LinearRegression()
    if log_target:
        model = TransformedTargetRegressor(regressor=model, func=np.log1p, inverse_func=np.expm1)
    return Pipeline([("preprocessor", preprocessor), ("regressor", model)])

print("Numeric features:", car_numeric_features)
print("Categorical features:", car_categorical_features)
print("Target: selling_price (INR)")


### Deduplicated train-test split — 80:20, random_state=42

The holdout is reserved for final evaluation. All feature and hyperparameter choices below use the training partition only.


In [ ]:
from sklearn.model_selection import train_test_split

X_car_train, X_car_test, y_car_train, y_car_test = train_test_split(
    X_car, y_car, test_size=0.20, random_state=42
)
print("Unique usable rows:", len(X_car))
print("Training rows:", len(X_car_train), "| Test rows:", len(X_car_test))


### Model comparison and tuning — training data only


In [ ]:
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.linear_model import Lasso, Ridge
from sklearn.model_selection import GridSearchCV, KFold, RandomizedSearchCV, cross_val_score

car_cv = KFold(n_splits=5, shuffle=True, random_state=42)

# Compare name-derived feature sets using the same log-target Linear Regression pipeline.
car_feature_configs = {
    "Brand + source categories": ["brand", "fuel", "seller_type", "transmission", "owner"],
    "Brand + model family": ["brand", "model_family", "fuel", "seller_type", "transmission", "owner"],
    "Brand + model family + full name": car_categorical_features,
}
car_feature_cv = {}
for label, categories in car_feature_configs.items():
    candidate = make_car_pipeline(log_target=True, categorical_features=categories)
    scores = cross_val_score(candidate, X_car_train, y_car_train, cv=car_cv, scoring="r2", n_jobs=1)
    car_feature_cv[label] = {"mean": float(scores.mean()), "std": float(scores.std(ddof=1))}

# Raw/log-target Linear Regression and regularized/tree candidates use the same cleaned features.
car_raw_linear = make_car_pipeline(log_target=False)
car_log_linear = make_car_pipeline(log_target=True)
car_raw_cv = cross_val_score(car_raw_linear, X_car_train, y_car_train, cv=car_cv, scoring="r2", n_jobs=1)
car_log_cv = cross_val_score(car_log_linear, X_car_train, y_car_train, cv=car_cv, scoring="r2", n_jobs=1)

car_ridge_search = GridSearchCV(
    make_car_pipeline(log_target=True, estimator=Ridge()),
    param_grid={"regressor__regressor__alpha": [0.1, 1, 10, 50, 100]},
    cv=car_cv, scoring="r2", n_jobs=1, refit=True,
)
car_ridge_search.fit(X_car_train, y_car_train)

car_lasso_search = GridSearchCV(
    make_car_pipeline(log_target=True, estimator=Lasso(max_iter=10000)),
    param_grid={"regressor__regressor__alpha": [0.0001, 0.001, 0.01, 0.1, 1]},
    cv=car_cv, scoring="r2", n_jobs=1, refit=True,
)
car_lasso_search.fit(X_car_train, y_car_train)

car_extra_trees_search = RandomizedSearchCV(
    make_car_pipeline(log_target=True, estimator=ExtraTreesRegressor(random_state=42, n_jobs=1)),
    param_distributions={
        "regressor__regressor__n_estimators": [60, 100, 140],
        "regressor__regressor__max_depth": [None, 12, 20],
        "regressor__regressor__min_samples_leaf": [1, 2, 4, 6],
        "regressor__regressor__max_features": [0.6, 0.8, 1.0],
    },
    n_iter=8, cv=car_cv, scoring="r2", random_state=42, n_jobs=1, refit=True,
)
car_extra_trees_search.fit(X_car_train, y_car_train)

car_cv_r2 = {
    "Raw Linear Regression": float(car_raw_cv.mean()),
    "Log-target Linear Regression": float(car_log_cv.mean()),
    "Ridge Regression (log target)": float(car_ridge_search.best_score_),
    "Lasso Regression (log target)": float(car_lasso_search.best_score_),
    "Extra Trees (log target)": float(car_extra_trees_search.best_score_),
}
car_cv_std = {
    "Raw Linear Regression": float(car_raw_cv.std(ddof=1)),
    "Log-target Linear Regression": float(car_log_cv.std(ddof=1)),
    "Ridge Regression (log target)": float(car_ridge_search.cv_results_["std_test_score"][car_ridge_search.best_index_]),
    "Lasso Regression (log target)": float(car_lasso_search.cv_results_["std_test_score"][car_lasso_search.best_index_]),
    "Extra Trees (log target)": float(car_extra_trees_search.cv_results_["std_test_score"][car_extra_trees_search.best_index_]),
}
car_model_candidates = {
    "Raw Linear Regression": car_raw_linear,
    "Log-target Linear Regression": car_log_linear,
    "Ridge Regression (log target)": car_ridge_search.best_estimator_,
    "Lasso Regression (log target)": car_lasso_search.best_estimator_,
    "Extra Trees (log target)": car_extra_trees_search.best_estimator_,
}
car_best_name = max(car_cv_r2, key=car_cv_r2.get)
car_best_pipeline = car_model_candidates[car_best_name]
car_best_pipeline.fit(X_car_train, y_car_train)

print("Feature-set comparison (mean training CV R² ± SD):")
display(pd.DataFrame(car_feature_cv).T.rename(columns={"mean": "Mean CV R²", "std": "CV SD"}).style.format("{:.3f}"))
print("Model selection comparison (training-only 5-fold CV R²):")
display(pd.Series(car_cv_r2, name="Mean CV R²").to_frame().style.format("{:.3f}"))
print("Ridge alpha:", car_ridge_search.best_params_, "| Lasso alpha:", car_lasso_search.best_params_)
print("Extra Trees search best parameters:", car_extra_trees_search.best_params_)
print("Selected without using the held-out test set:", car_best_name)
print("Selected model training-CV R² ± SD:", f"{car_cv_r2[car_best_name]:.3f} ± {car_cv_std[car_best_name]:.3f}")


### Final held-out model evaluation

After selecting the model using only training-fold scores, fit each reported candidate on the complete training partition. The test data is used here for final reporting, not for model selection.


In [ ]:
def car_regression_metrics(actual, predicted):
    mse = mean_squared_error(actual, predicted)
    return {
        "MAE": mean_absolute_error(actual, predicted),
        "MSE": mse,
        "RMSE": np.sqrt(mse),
        "R2": r2_score(actual, predicted),
    }

car_model_candidates["Raw Linear Regression"].fit(X_car_train, y_car_train)
car_model_candidates["Log-target Linear Regression"].fit(X_car_train, y_car_train)
car_predictions = {
    name: model.predict(X_car_test)
    for name, model in car_model_candidates.items()
}
car_metrics = pd.DataFrame({
    name: car_regression_metrics(y_car_test, predicted)
    for name, predicted in car_predictions.items()
}).T
car_best_predictions = car_predictions[car_best_name]

display(car_metrics.style.format({"MAE": "{:,.2f}", "MSE": "{:,.2f}", "RMSE": "{:,.2f}", "R2": "{:.3f}"}))
print("Selected using training-only CV:", car_best_name)
print("R² ≥ 0.75 assignment target:", "MET" if car_metrics.loc[car_best_name, "R2"] >= 0.75 else "NOT MET")


### Actual-versus-predicted and residual analysis


In [ ]:
car_residuals = y_car_test.to_numpy() - car_best_predictions
plot_low = min(float(y_car_test.min()), float(np.min(car_best_predictions)))
plot_high = max(float(y_car_test.max()), float(np.max(car_best_predictions)))
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.scatterplot(x=y_car_test, y=car_best_predictions, alpha=0.55, s=24, ax=axes[0])
axes[0].plot([plot_low, plot_high], [plot_low, plot_high], "--", color="crimson", label="Ideal prediction")
axes[0].set(title=f"Actual vs predicted — {car_best_name}", xlabel="Actual selling price (INR)", ylabel="Predicted selling price (INR)")
axes[0].legend()

sns.scatterplot(x=y_car_test, y=car_residuals, alpha=0.5, s=24, ax=axes[1])
axes[1].axhline(0, color="crimson", linestyle="--")
axes[1].set(title="Residuals vs actual price", xlabel="Actual selling price (INR)", ylabel="Residual (INR)")

sns.histplot(car_residuals, bins=35, color="slateblue", ax=axes[2])
axes[2].axvline(0, color="crimson", linestyle="--")
axes[2].set(title="Residual distribution", xlabel="Residual (INR)", ylabel="Held-out listings")
plt.tight_layout()
plt.show()
print("Residual = actual selling price − predicted selling price.")


### Results interpretation and limitations


In [ ]:
selected_car_metrics = car_metrics.loc[car_best_name]
print(f"Selected model: {car_best_name} | test R²={selected_car_metrics['R2']:.3f}, MAE=₹{selected_car_metrics['MAE']:,.0f}, RMSE=₹{selected_car_metrics['RMSE']:,.0f}.")
print("The selected model meets the assignment target only if its held-out R² is at least 0.75; see the measured table above.")
print("The historical pre-optimization run reported R² ≈ 0.519 with duplicate rows retained and simpler brand features. Preserve it as a historical baseline; the new deduplicated feature/model pipeline is not a like-for-like rerun.")
print("Important omitted predictors include vehicle condition, trim, accident history, and location; these limit price estimates.")
print("Do not interpret the best test R² as the selection criterion: the selected candidate is the one with the strongest training-only cross-validation R².")


### Save the selected pipeline and inspect held-out prediction examples

Examples below are actual rows from the reserved test partition. Predictions come from the selected pipeline fitted only on training rows.


In [ ]:
from pathlib import Path

car_model_path = Path("car_price_best_pipeline.joblib")
joblib.dump(car_best_pipeline, car_model_path)
car_loaded_pipeline = joblib.load(car_model_path)
car_examples = df_car.loc[X_car_test.index, ["name", "year", "km_driven", "selling_price"]].copy()
car_examples = car_examples.iloc[:3].rename(columns={"selling_price": "Actual selling_price"})
car_examples["Predicted selling_price"] = car_loaded_pipeline.predict(X_car_test.loc[car_examples.index])
car_examples["Residual"] = car_examples["Actual selling_price"] - car_examples["Predicted selling_price"]
print("Saved and reloaded:", car_model_path)
display(car_examples.style.format({"Actual selling_price": "₹{:,.0f}", "Predicted selling_price": "₹{:,.0f}", "Residual": "₹{:,.0f}"}))

car_feature_names = car_best_pipeline.named_steps["preprocessor"].get_feature_names_out()
car_estimator = car_best_pipeline.named_steps["regressor"].regressor_
car_coefficients = pd.DataFrame({"feature": car_feature_names, "coefficient": car_estimator.coef_})
car_coefficients["absolute_coefficient"] = car_coefficients["coefficient"].abs()
print("Largest absolute coefficients (log-price scale; categorical effects are relative to an encoded reference):")
display(car_coefficients.nlargest(10, "absolute_coefficient")[["feature", "coefficient"]].style.format({"coefficient": "{:.4f}"}))


### Section conclusion

The selected model is chosen using training-only cross-validation, then evaluated once on the held-out test split. The dashboard and presentation report the same measured metrics and preserve the original R² ≈ 0.519 result as a historical, non-comparable baseline. Even with the optimized pipeline, predictions remain limited by missing vehicle-condition, trim, accident, and location data.


<a id="part-1-group-c-mall-customer-clustering"></a>
# PART 1 - GROUP C: MALL CUSTOMER CLUSTERING
## Selected case K2 - Mall Customer Segmentation

### Objective

Segment customers using only annual income and spending score. Compare inertia for K=1 through K=10, fit the required K=5 K-Means model, and inspect its silhouette score and centers.

In [ ]:
import io
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid", palette="deep")

### Dataset upload - Mall Customers CSV

Run in Colab and select **Mall_Customers.csv**.

In [ ]:
# Colab upload or local VS Code file picker; select Mall_Customers.csv.
from pathlib import Path

try:
    from google.colab import files
    uploaded = files.upload()
except ImportError:
    from tkinter import Tk, filedialog

    picker = Tk()
    picker.withdraw()
    selected_file = filedialog.askopenfilename(
        title="Select Mall_Customers.csv",
        filetypes=[("CSV files", "*.csv")],
    )
    picker.destroy()
    if not selected_file:
        raise FileNotFoundError("No CSV was selected.")
    uploaded = {Path(selected_file).name: Path(selected_file).read_bytes()}

expected_filename = 'Mall_Customers.csv'
if expected_filename not in uploaded:
    raise FileNotFoundError(
        f"Expected {expected_filename!r}. Rerun this cell and upload/select that exact CSV filename."
    )
df_mall_raw = pd.read_csv(
    io.BytesIO(uploaded[expected_filename])
)
print("Loaded:", expected_filename)

### Load and inspect

In [ ]:
print("Shape:", df_mall_raw.shape)
print("Columns:", df_mall_raw.columns.tolist())
display(df_mall_raw.dtypes.rename("dtype").to_frame())
display(df_mall_raw.isna().sum().rename("missing").to_frame())
display(df_mall_raw.head())

### Data cleaning and feature selection

Validate and convert the two clustering features to numeric; exclude rows missing either. CustomerID, Gender, and Age are not clustering inputs. This is unsupervised learning, so no artificial supervised split is used.

In [ ]:
mall_features = ["Annual Income (k$)", "Spending Score (1-100)"]
mall_required = [*mall_features, "CustomerID", "Gender", "Age"]
mall_missing_columns = sorted(set(mall_required) - set(df_mall_raw.columns))
if mall_missing_columns:
    raise ValueError(f"Mall CSV is missing required columns: {mall_missing_columns}")
df_mall = df_mall_raw.copy()
for column in mall_features:
    df_mall[column] = pd.to_numeric(df_mall[column], errors="coerce")
rows_before = len(df_mall)
df_mall = df_mall.dropna(subset=mall_features).copy()
print("Rows removed for missing/invalid clustering fields:", rows_before - len(df_mall))
X_mall = df_mall[mall_features].copy()
print("Features used:", X_mall.columns.tolist())
print("Excluded: CustomerID, Gender, Age")

### Exploratory data analysis

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
sns.scatterplot(data=df_mall, x=mall_features[0], y=mall_features[1], s=65, ax=ax)
ax.set(title="Mall Customers Before Clustering", xlabel="Annual income (thousands)", ylabel="Spending score (1-100)")
plt.tight_layout()
plt.show()
display(X_mall.describe().T)

### Data preprocessing

Standardize the two features before K-Means so income units do not dominate the spending-score scale.

In [ ]:
mall_scaler = StandardScaler()
X_mall_scaled = mall_scaler.fit_transform(X_mall)
print("Scaled means:", np.round(X_mall_scaled.mean(axis=0), 3))
print("Scaled standard deviations:", np.round(X_mall_scaled.std(axis=0), 3))

### Elbow Method - inertia for K=1 to K=10

In [ ]:
mall_inertia = {}
for k in range(1, 11):
    candidate = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X_mall_scaled)
    mall_inertia[k] = candidate.inertia_
display(pd.DataFrame({"K": list(mall_inertia), "Inertia": list(mall_inertia.values())}))
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(list(mall_inertia), list(mall_inertia.values()), marker="o")
ax.set(title="Elbow Method for Mall Customer Clustering", xlabel="Number of clusters (K)", ylabel="Inertia")
ax.set_xticks(range(1, 11))
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()
print("The required final model uses K=5; assess the elbow visually and silhouette below.")

### Final K-Means model - K=5 and silhouette evaluation

In [ ]:
mall_kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
mall_labels = mall_kmeans.fit_predict(X_mall_scaled)
mall_silhouette = silhouette_score(X_mall_scaled, mall_labels)
mall_centers = mall_scaler.inverse_transform(mall_kmeans.cluster_centers_)
df_mall["Cluster"] = mall_labels
print(f"K=5 silhouette score: {mall_silhouette:.3f}")
display(pd.DataFrame(mall_centers, columns=mall_features).round(2))

### Required plot - five clusters and centroids

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
sns.scatterplot(data=df_mall, x=mall_features[0], y=mall_features[1], hue="Cluster", palette="tab10", s=65, ax=ax)
ax.scatter(mall_centers[:, 0], mall_centers[:, 1], marker="X", s=260, c="black", edgecolor="white", label="Centroids", zorder=5)
ax.set(title="Mall Customer Segments (K=5)", xlabel="Annual income (thousands)", ylabel="Spending score (1-100)")
ax.legend(title="Cluster / center", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

### Cluster sizes, names, and interpretation

In [ ]:
def name_mall_cluster(income, spending):
    income_level = "Low income" if income < 40 else "High income" if income > 75 else "Medium income"
    spending_level = "low spending" if spending < 40 else "high spending" if spending > 60 else "moderate spending"
    return f"{income_level}, {spending_level}"

mall_cluster_names = {i: name_mall_cluster(*mall_centers[i]) for i in range(5)}
df_mall["Segment"] = df_mall["Cluster"].map(mall_cluster_names)
mall_segment_summary = df_mall.groupby(["Cluster", "Segment"], sort=True).agg(
    Customers=(mall_features[0], "size"),
    Mean_Annual_Income=(mall_features[0], "mean"),
    Mean_Spending_Score=(mall_features[1], "mean"),
).round(2)
display(mall_segment_summary)
for cluster in range(5):
    print(f"Cluster {cluster}: {mall_cluster_names[cluster]}.")

### Save scaler/model and assign a new customer

In [ ]:
mall_scaler_path, mall_model_path = "mall_customer_scaler.joblib", "mall_customer_kmeans.joblib"
joblib.dump(mall_scaler, mall_scaler_path)
joblib.dump(mall_kmeans, mall_model_path)
mall_loaded_scaler, mall_loaded_model = joblib.load(mall_scaler_path), joblib.load(mall_model_path)
new_mall_customer = pd.DataFrame([[70, 60]], columns=mall_features)
new_mall_cluster = int(mall_loaded_model.predict(mall_loaded_scaler.transform(new_mall_customer))[0])
print("Saved:", mall_scaler_path, "and", mall_model_path)
display(new_mall_customer)
print(f"Assigned to Cluster {new_mall_cluster}: {mall_cluster_names[new_mall_cluster]}.")

### Section conclusion

The elbow curve and silhouette score describe the actual five-cluster fit. Segment names summarize fitted centers, not ground-truth labels. The scaler and model are saved separately and reloaded for the new-customer example.

<a id="part-2-student-performance-prediction"></a>
# PART 2 - SELF-CHOSEN PROJECT: STUDENT PERFORMANCE PREDICTION
## Regression project - Predicting `Exam_Score`

### Problem statement (exactly two lines)

This project predicts a student's `Exam_Score` from recorded study habits and contextual factors.  
An early estimate can help educators identify patterns and plan support, but it does not determine or guarantee an individual result.

**Why it is useful:** A data-informed estimate can support discussion about study patterns and learning resources; it does not replace educators' judgment.

In [ ]:
import io
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

sns.set_theme(style="whitegrid", palette="deep")

### Dataset upload - Student Performance Factors CSV

Run in Colab and select **StudentPerformanceFactors.csv**.

In [ ]:
# Colab upload or local VS Code file picker; select StudentPerformanceFactors.csv.
from pathlib import Path

try:
    from google.colab import files
    uploaded = files.upload()
except ImportError:
    from tkinter import Tk, filedialog

    picker = Tk()
    picker.withdraw()
    selected_file = filedialog.askopenfilename(
        title="Select StudentPerformanceFactors.csv",
        filetypes=[("CSV files", "*.csv")],
    )
    picker.destroy()
    if not selected_file:
        raise FileNotFoundError("No CSV was selected.")
    uploaded = {Path(selected_file).name: Path(selected_file).read_bytes()}

expected_filename = 'StudentPerformanceFactors.csv'
if expected_filename not in uploaded:
    raise FileNotFoundError(
        f"Expected {expected_filename!r}. Rerun this cell and upload/select that exact CSV filename."
    )
df_student_raw = pd.read_csv(
    io.BytesIO(uploaded[expected_filename])
)
print("Loaded:", expected_filename)

### Dataset source, target, shape, and feature descriptions

**Dataset:** Student Performance Factors (`StudentPerformanceFactors.csv`)  
**Source:** [Kaggle - Student Performance Factors](https://www.kaggle.com/datasets/lainguyn123/student-performance-factors)  
**Target:** `Exam_Score`

| Feature | Description |
|---|---|
| Hours_Studied | Hours spent studying |
| Attendance | Attendance percentage |
| Parental_Involvement | Reported parental involvement level |
| Access_to_Resources | Reported access to learning resources |
| Extracurricular_Activities | Participation in extracurricular activities |
| Sleep_Hours | Average daily sleep hours |
| Previous_Scores | Prior academic score |
| Motivation_Level | Reported motivation level |
| Internet_Access | Whether internet access is available |
| Tutoring_Sessions | Number of tutoring sessions attended |
| Family_Income | Reported family income band |
| Teacher_Quality | Reported teacher quality |
| School_Type | Public or private school |
| Peer_Influence | Reported peer influence |
| Physical_Activity | Reported physical-activity measure |
| Learning_Disabilities | Reported status |
| Parental_Education_Level | Reported parental education level |
| Distance_from_Home | Reported distance band |
| Gender | Reported gender category |

The uploaded dataset's actual row/column count and names are displayed below.

In [ ]:
print("Dataset:", expected_filename, "| Uploaded shape:", df_student_raw.shape)
if df_student_raw.shape == (6607, 20):
    print("Reference dataset shape: 6,607 rows × 20 columns.")
print("Columns:", df_student_raw.columns.tolist())
display(df_student_raw.dtypes.rename("dtype").to_frame())
display(df_student_raw.head())

### Data inspection and cleaning

In [ ]:
student_features = [
    "Hours_Studied", "Attendance", "Parental_Involvement", "Access_to_Resources",
    "Extracurricular_Activities", "Sleep_Hours", "Previous_Scores", "Motivation_Level",
    "Internet_Access", "Tutoring_Sessions", "Family_Income", "Teacher_Quality",
    "School_Type", "Peer_Influence", "Physical_Activity", "Learning_Disabilities",
    "Parental_Education_Level", "Distance_from_Home", "Gender",
]
student_target = "Exam_Score"
student_required = student_features + [student_target]
missing_columns = sorted(set(student_required) - set(df_student_raw.columns))
if missing_columns:
    raise ValueError(f"Student CSV is missing required columns: {missing_columns}")
print("Missing values:")
display(df_student_raw[student_required].isna().sum().rename("missing").to_frame())
print("Exact duplicate rows (retained):", int(df_student_raw.duplicated().sum()))
df_student = df_student_raw[student_required].copy()
student_numeric = ["Hours_Studied", "Attendance", "Sleep_Hours", "Previous_Scores", "Tutoring_Sessions", "Physical_Activity", student_target]
for column in student_numeric:
    df_student[column] = pd.to_numeric(df_student[column], errors="coerce")
rows_before = len(df_student)
df_student = df_student.dropna(subset=[student_target]).copy()
print("Rows removed for missing target:", rows_before - len(df_student))
print("Missing predictor values will be imputed within the training-fitted pipeline.")
for column in student_numeric:
    values = df_student[column].dropna()
    q1, q3 = values.quantile([0.25, 0.75])
    low, high = q1 - 1.5 * (q3 - q1), q3 + 1.5 * (q3 - q1)
    print(f"{column}: {int(((values < low) | (values > high)).sum())} IQR outlier(s); retained.")

### Exploratory data analysis - four visualizations

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
sns.histplot(data=df_student, x="Exam_Score", bins=25, kde=True, color="steelblue", ax=axes[0, 0])
axes[0, 0].set(title="Exam Score Distribution", xlabel="Exam score", ylabel="Students")
sns.scatterplot(data=df_student, x="Hours_Studied", y="Exam_Score", hue="Attendance", palette="viridis", alpha=0.65, ax=axes[0, 1])
axes[0, 1].set(title="Study Hours, Attendance, and Score", xlabel="Hours studied", ylabel="Exam score")
sns.boxplot(data=df_student, x="Parental_Involvement", y="Exam_Score", order=["Low", "Medium", "High"], ax=axes[1, 0])
axes[1, 0].set(title="Exam Score by Parental Involvement", xlabel="Parental involvement", ylabel="Exam score")
student_numeric_columns = df_student.select_dtypes(include=[np.number]).columns.tolist()
sns.heatmap(df_student[student_numeric_columns].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=axes[1, 1])
axes[1, 1].set(title="Correlation Heatmap - Numeric Variables")
plt.tight_layout()
plt.show()

### Plot observations

Describe the distributions and group differences after viewing the generated plots. Correlations and group differences are descriptive associations, not evidence that a factor causes score changes.

In [ ]:
print("Observed Exam_Score range:", df_student["Exam_Score"].min(), "to", df_student["Exam_Score"].max())
display(df_student.groupby("Parental_Involvement", dropna=False)["Exam_Score"].mean().rename("mean Exam_Score").to_frame())
student_corr = df_student[student_numeric_columns].corr()["Exam_Score"].drop("Exam_Score").sort_values(key=abs, ascending=False)
print("Numeric correlations with Exam_Score (descriptive, not causal):")
display(student_corr.rename("correlation").to_frame())

### Data preprocessing

Median-impute and standardize numeric inputs; most-frequent-impute and one-hot encode categorical inputs. These transformations are fitted on training data only.

In [ ]:
X_student = df_student[student_features].copy()
y_student = df_student[student_target].copy()
student_numeric_features = X_student.select_dtypes(include=[np.number]).columns.tolist()
student_categorical_features = [c for c in student_features if c not in student_numeric_features]

def make_student_pipeline(regressor):
    preprocessor = ColumnTransformer([
        ("numeric", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), student_numeric_features),
        ("categorical", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]), student_categorical_features),
    ])
    return Pipeline([("preprocessor", preprocessor), ("regressor", regressor)])

print("Numeric inputs:", student_numeric_features)
print("Categorical inputs:", student_categorical_features)
print("Total required input features:", len(student_features))

### Train-test split - 80:20, random_state=42

In [ ]:
X_student_train, X_student_test, y_student_train, y_student_test = train_test_split(
    X_student, y_student, test_size=0.20, random_state=42
)
print("Training rows:", len(X_student_train), "| Test rows:", len(X_student_test))

### Model training - Linear Regression and Random Forest

In [ ]:
from sklearn.model_selection import KFold, cross_val_score

student_models = {
    "Linear Regression": make_student_pipeline(LinearRegression()),
    "Random Forest Regressor": make_student_pipeline(
        RandomForestRegressor(n_estimators=150, random_state=42, n_jobs=1, min_samples_leaf=2)
    ),
}
student_cv = KFold(n_splits=5, shuffle=True, random_state=42)
student_cv_r2 = {
    name: cross_val_score(
        model, X_student_train, y_student_train, cv=student_cv, scoring="r2", n_jobs=1
    ).mean()
    for name, model in student_models.items()
}
student_best_name = max(student_cv_r2, key=student_cv_r2.get)
for model in student_models.values():
    model.fit(X_student_train, y_student_train)
student_best_pipeline = student_models[student_best_name]
print("Training-only 5-fold CV R2:")
display(pd.Series(student_cv_r2, name="Mean CV R2").to_frame().style.format("{:.3f}"))
print("Model selected without using the held-out test set:", student_best_name)

### Model evaluation and comparison

In [ ]:
def student_metrics(actual, predicted):
    mse = mean_squared_error(actual, predicted)
    return {
        "MAE": mean_absolute_error(actual, predicted),
        "MSE": mse,
        "RMSE": np.sqrt(mse),
        "R2": r2_score(actual, predicted),
    }

# Report both models on the held-out test set; do not use these results to select a model.
student_predictions = {
    name: model.predict(X_student_test) for name, model in student_models.items()
}
student_metrics_table = pd.DataFrame({
    name: student_metrics(y_student_test, predicted)
    for name, predicted in student_predictions.items()
}).T
display(student_metrics_table.style.format(
    {"MAE": "{:.3f}", "MSE": "{:.3f}", "RMSE": "{:.3f}", "R2": "{:.3f}"}
))
print("Selected using training-only 5-fold CV R2:", student_best_name)
print("The test metrics above are final evaluation only.")

### Actual versus predicted scores

In [ ]:
student_best_predictions = student_predictions[student_best_name]
fig, ax = plt.subplots(figsize=(7, 6))
sns.scatterplot(x=y_student_test, y=student_best_predictions, alpha=0.6, ax=ax)
low = min(float(y_student_test.min()), float(np.min(student_best_predictions)))
high = max(float(y_student_test.max()), float(np.max(student_best_predictions)))
ax.plot([low, high], [low, high], "--", color="crimson", label="Ideal prediction")
ax.set(title=f"Actual vs Predicted Exam Scores - {student_best_name}", xlabel="Actual Exam_Score", ylabel="Predicted Exam_Score")
ax.legend()
plt.tight_layout()
plt.show()

### Results interpretation

In [ ]:
print(f"{student_best_name} performed best on the fixed 20% test split.")
print("Study habits and prior scores can be useful predictive signals; interpret associations in the displayed plots cautiously.")
print("The data may omit individual, classroom, and assessment context; predictions are estimates, not guaranteed results.")

### Save/reload selected pipeline and predict three new students

In [ ]:
student_model_path = "student_performance_best_pipeline.joblib"
joblib.dump(student_best_pipeline, student_model_path)
student_loaded_model = joblib.load(student_model_path)
# Manually specified examples include all required inputs and are not test-set rows.
new_students = pd.DataFrame([
    [25, 95, "High", "High", "Yes", 8, 85, "High", "Yes", 3, "High", "High", "Private", "Positive", 5, "No", "Postgraduate", "Near", "Female"],
    [8, 70, "Medium", "Medium", "No", 7, 65, "Medium", "Yes", 1, "Medium", "Medium", "Public", "Neutral", 3, "No", "College", "Moderate", "Male"],
    [15, 85, "High", "Medium", "Yes", 8, 78, "High", "Yes", 2, "Medium", "High", "Public", "Positive", 4, "No", "High School", "Near", "Female"],
], columns=student_features)
student_new_scores = student_loaded_model.predict(new_students)
student_prediction_table = new_students.copy()
student_prediction_table.insert(0, "Example", ["Student 1", "Student 2", "Student 3"])
student_prediction_table["Predicted Exam_Score"] = np.round(student_new_scores, 2)
print("Saved and reloaded:", student_model_path)
display(student_prediction_table)

### Five-line project conclusion

In [ ]:
student_conclusion = [
    f"Linear Regression test R2 was {student_metrics_table.loc['Linear Regression', 'R2']:.3f}; Random Forest test R2 was {student_metrics_table.loc['Random Forest Regressor', 'R2']:.3f}.",
    f"{student_best_name} was selected using training-only cross-validation before final test evaluation.",
    "Study patterns and prior scores show predictive associations, not proof of cause.",
    "The dataset may omit important individual and classroom context, limiting generalization.",
    "More representative data and validated features may help; predictions are estimates, not guaranteed exam results.",
]
assert len(student_conclusion) == 5
print("\n".join(student_conclusion))

<a id="overall-conclusion"></a>
# OVERALL CONCLUSION

This notebook applies classification, regression, and unsupervised clustering to four independently uploaded datasets. Interpret each result using the metrics generated from the uploaded data; no performance value is assumed in advance. Iris accuracy and car-price R-squared targets are checked against held-out results. Preprocessing is fitted within training pipelines where applicable, and valid observations are not removed just to improve scores. Predictions are estimates and should be considered alongside source-data limitations.